<center>
<h1>Chirundu Town Council CDF and Financial Dataset</h1>
<b>CSC 4792 Group Project - Group 39</b><br/>
University of Zambia<br/>
September 2026
</center>

---

## Overview

This consolidated notebook documents approved and proposed CDF projects, the 2025 decision list, annual budgets, financial statements, performance records and procurement plans. Different record types stay in separate tables so applications, allocations, estimates and actual expenditure are not confused.

**Sources:** Council documents in `data/source_inventory/download_inventory.json`; member notebooks archived under `data/interim/member_notebooks/`.

**Format:** UTF-8 CSV, pipe (`|`) separator and `db-unza26-csc4792-` filename prefix as required by the assignment.

**Reference notebook:** The numbered sections and pandas first-look/missing-value style follow *Starter Notebook: CS1 Failure Prediction Dataset* by Lighton Phiri (July 2026), referenced in the existing notebook. Council data and extraction methods are specific to this project.

---


## 1. Environment Setup

We import libraries for tables, PDF extraction and OCR. Extraction functions and manual corrections are included in this notebook. The procurement helper is also available as a script.

For a full run, keep this notebook with the project folders `data/raw/council_documents/`, `data/source_inventory/` and `models/`. Run it from the project folder or its `notebooks` subfolder, using the project's Python environment. Install the packages listed in `src/requirements-cdf-pilot.txt`, including `xlrd` for the 2026 workbook, if needed. Downloaded PDFs and the English OCR model are still required; this notebook is self-contained in code, not in source data.

Run the cells from top to bottom. OCR takes longer than loading a CSV. The financial and performance sections reuse saved OCR when available; their functions can recreate it from the PDFs.

In [1]:
# Import libraries
from pathlib import Path
from html.parser import HTMLParser
from urllib.parse import urljoin
import json
import re
import pandas as pd
import pdfplumber
import pypdfium2 as pdfium
from pypdf import PdfReader
from rapidocr_onnxruntime import RapidOCR
from IPython.display import display

pd.set_option('display.max_colwidth', 70)

# Update ROOT if the project is stored somewhere else
ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent

print('All libraries imported successfully.')
print('Project folder:', ROOT)

All libraries imported successfully.
Project folder: C:\Users\Arthur F Chipeta\Desktop\chirundu-cdf-project


## 2. Collect Document Links and Load the Inventory

The council webpages were saved as HTML before extraction. We read their links, keep document downloads, combine duplicate URLs and record each source page. The parser below was previously in a separate script and is now part of this notebook.

The observed links were reviewed and organised into `download_inventory.json`, with a stable inventory number, category and filename for each selected source. This was a manual source-selection step. Files were downloaded using the supplied `data/source_inventory/download_documents.ps1`, which skips existing files. That collection script remains part of the codebase; it is not needed to run extraction once the files are present.

Re-running this section reads the saved pages rather than changing the collection to whatever is currently online. The URLs and page snapshots document where the files came from. The notebook does not require any local Python module.

The reference notebook starts by loading a finished CSV. Here we first create the CSVs from the source PDFs; section 8 loads the saved results for inspection.

In [2]:
class Links(HTMLParser):
    def __init__(self):
        super().__init__()
        self.links = []
        self.href = None
        self.parts = []
    def handle_starttag(self, tag, attrs):
        if tag == 'a':
            self.href = dict(attrs).get('href')
            self.parts = []
    def handle_data(self, data):
        if self.href is not None:
            self.parts.append(data)
    def handle_endtag(self, tag):
        if tag == 'a' and self.href is not None:
            self.links.append((' '.join(' '.join(self.parts).split()), self.href))
            self.href = None

In [3]:
# Read links from the saved official webpages
var_source_folder = ROOT / 'data/source_inventory'
var_base_url = 'https://www.chirunducouncil.gov.zm/'
var_observed = {}

for var_page_path in sorted(var_source_folder.glob('*.html')):
    var_page_id = '959' if var_page_path.stem == 'publications' else var_page_path.stem.removeprefix('page-')
    var_source_url = var_base_url + '?page_id=' + var_page_id
    var_parser = Links()
    var_parser.feed(var_page_path.read_text(encoding='utf-8-sig'))

    for var_label, var_href in var_parser.links:
        var_url = urljoin(var_source_url, var_href)
        if '/wp-content/uploads/' not in var_url:
            continue
        if not var_url.lower().split('?')[0].endswith(('.pdf', '.xls', '.xlsx', '.doc', '.docx', '.csv')):
            continue
        var_key = var_url.removeprefix('http://').removeprefix('https://')
        if var_key not in var_observed:
            var_observed[var_key] = {'url': var_url, 'labels': [], 'source_pages': []}
        if var_label and var_label not in var_observed[var_key]['labels']:
            var_observed[var_key]['labels'].append(var_label)
        if var_source_url not in var_observed[var_key]['source_pages']:
            var_observed[var_key]['source_pages'].append(var_source_url)

var_links_df = pd.DataFrame(var_observed.values())
print('Unique document links found:', len(var_links_df))
display(var_links_df.head())

Unique document links found: 104


,url,labels,source_pages
0,http://www.chirunducouncil.gov.zm/wp-content/uploads/2024/11/2022-...,[2022-Approved-Community-Projects],[https://www.chirunducouncil.gov.zm/?page_id=3241]
1,http://www.chirunducouncil.gov.zm/wp-content/uploads/2024/11/2023-...,[2023-Approved-Community-Projects],[https://www.chirunducouncil.gov.zm/?page_id=3241]
2,http://www.chirunducouncil.gov.zm/wp-content/uploads/2024/11/2024-...,[2024-Approved-Community-Projects],[https://www.chirunducouncil.gov.zm/?page_id=3241]
3,http://www.chirunducouncil.gov.zm/wp-content/uploads/2025/06/2025-...,[2025-Approved-Community-Projects],[https://www.chirunducouncil.gov.zm/?page_id=3241]
4,http://www.chirunducouncil.gov.zm/wp-content/uploads/2026/05/2026-...,[2026 Approved Community Projects],[https://www.chirunducouncil.gov.zm/?page_id=3241]


In [4]:
var_inventory_file = ROOT / 'data/source_inventory/download_inventory.json'
var_inventory = json.loads(var_inventory_file.read_text(encoding='utf-8'))
var_inventory_df = pd.DataFrame(var_inventory)
var_selected_ids = [1, 2, 3, 59, 5, 104, 34, 33, 32, 30, 39, 38, 35, 31, 36, 75]
display(var_inventory_df.loc[var_inventory_df['inventory_id'].isin(var_selected_ids),
                             ['inventory_id', 'display_title', 'url']])

,inventory_id,display_title,url
0,1,2022-Approved-Community-Projects,http://www.chirunducouncil.gov.zm/wp-content/uploads/2024/11/2022-...
1,2,2023-Approved-Community-Projects,http://www.chirunducouncil.gov.zm/wp-content/uploads/2024/11/2023-...
2,3,2024-Approved-Community-Projects,http://www.chirunducouncil.gov.zm/wp-content/uploads/2024/11/2024-...
4,5,2026 Approved Community Projects,http://www.chirunducouncil.gov.zm/wp-content/uploads/2026/05/2026-...
9,59,2025 CDF approved projects,http://www.chirunducouncil.gov.zm/wp-content/uploads/2025/11/2025-...
16,30,2026 OBB Budget K112.4Mn,http://www.chirunducouncil.gov.zm/wp-content/uploads/2026/02/2026-...
17,31,Bi Annual Performance Report,http://www.chirunducouncil.gov.zm/wp-content/uploads/2025/12/2025-...
18,32,Chirundu 2025 OBB Annual Budget Final - (K103.9 million),http://www.chirunducouncil.gov.zm/wp-content/uploads/2025/11/Chiru...
19,33,Chirundu Town Council OBB Budget 2024 (Final),http://www.chirunducouncil.gov.zm/wp-content/uploads/2024/08/Chiru...
20,34,Chirundu Town Council OBB Budget 2023 (Final),http://www.chirunducouncil.gov.zm/wp-content/uploads/2024/11/Chiru...


## 3. Extract and Clean the Approved CDF Project Lists

We begin with the detailed 2025 list, then repeat the method for 2022, 2023, 2024 and 2026. These are entries in annual approved lists, not necessarily different physical projects.

### 3.1 Read the 2025 page

The PDF is scanned, so we render its page as an image. OCR returns text and its positions. The positions let us put the text back into table rows and columns.

In [5]:
source_url = 'http://www.chirunducouncil.gov.zm/wp-content/uploads/2025/11/2025-Approved-Community-Projects-.pdf'

pdf_path = ROOT / 'data/raw/council_documents/01_cdf_community_projects/059--2025-Approved-Community-Projects-.pdf'
folder = ROOT / 'data/interim/ocr_2025'
folder.mkdir(parents=True, exist_ok=True)
pdf = pdfium.PdfDocument(pdf_path)
page_image = pdf[0].render(scale=2).to_pil()
image_path = folder / 'page_1.png'
page_image.save(image_path)
pdf.close()

print('PDF text:', PdfReader(pdf_path).pages[0].extract_text())
print('Image size:', page_image.size)

PDF text: CamScanner

Image size: (1684, 1190)


In [6]:
engine = RapidOCR(rec_model_path=str(ROOT / 'models/en_PP-OCRv3_rec_infer.onnx'), intra_op_num_threads=2, inter_op_num_threads=2, det_limit_side_len=1684)
result, elapsed = engine(str(image_path))
if not result:
    raise ValueError('OCR did not find any text on the page.')
(folder / 'ocr_text.json').write_text(json.dumps(result, indent=2, ensure_ascii=False), encoding='utf-8')

print('Text sections found:', len(result))
for box, text, score in result[:10]:
    print(text)

Text sections found: 114
2025 CDF Approved Community Projects
S.No
Project Name
Ward
Zone
Location
Comment
Construction of Solar powered
Ibbwemunyama Rural
1


### 3.2 Put OCR text into rows

The boundaries below were measured from the page rendered at scale 2. They describe the table layout, not the project values. A small position adjustment follows the slope of the scanned table.

In [7]:
# Pixel boundaries for the six columns and fourteen rows on this page.
columns = [250, 572, 738, 920, 1128, 1466]
row_edges = [154, 211, 293, 321, 377, 463, 545, 574, 659, 743, 773, 860, 918, 947, 1028]
names = ['project_name', 'ward', 'zone', 'location', 'approval_comment']
rows = []
for index, (top, bottom) in enumerate(zip(row_edges[:-1], row_edges[1:]), 1):
    row = {'project_no': index}
    for column, left, right in zip(names, columns[:-1], columns[1:]):
        words = []
        for box, text, confidence in result:
            x = sum(point[0] for point in box) / 4
            y = sum(point[1] for point in box) / 4
            # The scanned grid slopes slightly upwards towards the right.
            y += 14 * max(0, y - 150) / 878 * max(0, x - 250) / 1216
            if left <= x < right and top <= y < bottom:
                words.append((y, x, text))
        row[column] = ' '.join(word[2] for word in sorted(words)) or None
    rows.append(row)
df = pd.DataFrame(rows)
df.to_csv(folder / 'ocr_table_raw.csv', sep='|', index=False)
raw_df = df.copy()
display(raw_df.head())

,project_no,project_name,ward,zone,location,approval_comment
0,1,Construction of Solar powered borehole,Ibbwemunyama,Ibbwemunyama,Ibbwemunyama Rural Health Post,Approved
1,2,Community water project,Chirundu west,Chibende and Lusumpuko,Chibende and Lusumpuko Communities,Approved
2,3,Completion of Maternity Wing,Chirundu West,Chibende,Chibende,Approved
3,4,Construction of a 1x3 classroom block,Kapululira,Farao,Farao Community School,Approved
4,5,Construction of water borne toilets and solar powered borehole,Njame,Chibulameenda,NaN,Approved


### 3.3 Clean the 2025 records

We trim extra spaces, remove duplicates and apply the corrections read from the source image. OCR joined some cells and included stamp text. These manual corrections remain visible so the cleaning can be explained.

The approval comment for project 11 permits only the motor grader. No project amounts or physical implementation statuses are stated in this list, so they remain missing.

In [8]:
df = raw_df.copy()
for column in names:
    df[column] = df[column].astype('string').str.replace(r'\s+', ' ', regex=True).str.strip()

# Corrections read from the source page.
corrections = {
    (6, 'project_name'): 'Construction of a Double Storey Classroom Block (3 classes per storey with Offices)',
    (6, 'ward'): 'Ngombe Illede',
    (6, 'location'): 'Proposed Ngombe Illede Sec. School',
    (9, 'location'): '10 – Sikoongo zone, 04-Siachibubba zone and 06-T-Junction zone',
    (11, 'approval_comment'): 'Approved procurement of Motor grader',
    (12, 'approval_comment'): 'Approved',
    (13, 'approval_comment'): 'Approved',
    (14, 'approval_comment'): 'Approved',
}
for (project_no, column), value in corrections.items():
    df.loc[df['project_no'] == project_no, column] = value

df['ward'] = df['ward'].str.title()
df = df.drop_duplicates(subset=names).copy()
display(df.head())

,project_no,project_name,ward,zone,location,approval_comment
0,1,Construction of Solar powered borehole,Ibbwemunyama,Ibbwemunyama,Ibbwemunyama Rural Health Post,Approved
1,2,Community water project,Chirundu West,Chibende and Lusumpuko,Chibende and Lusumpuko Communities,Approved
2,3,Completion of Maternity Wing,Chirundu West,Chibende,Chibende,Approved
3,4,Construction of a 1x3 classroom block,Kapululira,Farao,Farao Community School,Approved
4,5,Construction of water borne toilets and solar powered borehole,Njame,Chibulameenda,<NA>,Approved


In [9]:
df['year'] = 2025
df['amount_zmw'] = pd.NA
df['implementation_status'] = pd.NA
df['source_url'] = source_url
df['source_page'] = 1
df = df[['project_no', 'year', 'project_name', 'ward', 'zone', 'location',
         'approval_comment', 'amount_zmw', 'implementation_status', 'source_url', 'source_page']]


print('Cleaned 2025 records:', len(df))

Cleaned 2025 records: 14


### 3.4 Set the layouts for the other years

The tables change between PDFs. `CDF_LAYOUTS` gives their column boundaries and numbered rows. The second 2023 page needs a 180-degree rotation, and the 2026 ward headings are excluded. These measurements must change if a different document is used.

In [10]:
def numbered(edges, first=1):
    return [(first+i, a, b) for i, (a, b) in enumerate(zip(edges[:-1], edges[1:]))]

CDF_LAYOUTS = [
    dict(year=2022, source_id=1, page=1, rotate=0,
         columns={'project_name':(105,389),'project_description':(389,805),'sector':(805,904),'ward':(1084,1183),'zone':(1183,1333),'location':(1333,1596)},
         rows=numbered([278,348,400,435,469,509,559,596,628,681,739,806,858,945,986,1056,1093]), tilt=(7,-0.01,278)),
    dict(year=2023, source_id=2, page=1, rotate=0,
         columns={'project_name':(104,523),'project_description':(523,759),'sector':(759,964),'ward':(1082,1202),'zone':(1202,1350),'location':(1350,1625)},
         rows=numbered([179,209,256,317,365,395,426,506,597,644,704,765,826,887,947,1023,1131]), tilt=(14,-0.016,179)),
    dict(year=2023, source_id=2, page=2, rotate=180,
         columns={'project_name':(120,534),'project_description':(534,770),'sector':(770,973),'ward':(1090,1211),'zone':(1211,1360),'location':(1360,1644)},
         rows=numbered([217,404,510,619],17), tilt=(-8,0,217)),
    dict(year=2024, source_id=3, page=1, rotate=0,
         columns={'project_name':(160,855),'sector':(855,1113),'ward':(1503,1618)},
         rows=numbered([188,230,279,311,344,391,455,519,600,713,825,867,926,944,961,978]), tilt=(8,-0.034,188)),
    dict(year=2026, source_id=5, page=1, rotate=0,
         columns={'project_name':(258,502),'ward':(502,734),'zone':(734,874),'location':(874,1059),'work_item':(1059,1518)},
         rows=[(1,200,360),(2,395,529),(3,559,641),(4,670,751),(5,782,866),(6,895,976)], tilt=(0,0,0)),
    dict(year=2026, source_id=5, page=2, rotate=0,
         columns={'project_name':(261,502),'ward':(502,731),'zone':(731,870),'location':(870,1057),'work_item':(1057,1523)},
         rows=[(7,96,170),(8,198,252),(9,281,338),(10,368,449),(11,482,599),(12,645,730),(13,734,788),(14,821,935),(15,937,1048)], tilt=(-18,0,0)),
    dict(year=2026, source_id=5, page=3, rotate=0,
         columns={'project_name':(197,475),'ward':(475,738),'zone':(738,891),'location':(891,1105),'work_item':(1105,1608)},
         rows=[(16,130,224),(17,225,287),(18,319,378),(19,409,443)], tilt=(-23,0,0)),
]

### 3.5 Extract the remaining lists

The first function groups words using a row's measured boundaries. The second renders each selected page, runs OCR and collects its records. We locate PDFs by their inventory prefix so the code does not depend on the original computer's absolute file paths.

In [11]:
def group_page(result, layout, source_url):
    rows = []
    for number, top, bottom in layout['rows']:
        row = {'project_no':number, 'year':layout['year'], 'source_url':source_url, 'source_page':layout['page']}
        for name, (left,right) in layout['columns'].items():
            words = []
            for box,text,score in result:
                x = sum(p[0] for p in box)/4
                y = sum(p[1] for p in box)/4
                offset, slope, origin = layout['tilt']
                y -= (offset+slope*(y-origin))*max(0,x-100)/1500
                if left <= x < right and top <= y < bottom:
                    # Bucket nearly level text together, then read left to right.
                    words.append((round(y/10),x,text))
            row[name] = ' '.join(w[2] for w in sorted(words)) or None
        rows.append(row)
    return rows

In [12]:
def extract_remaining_years():
    inventory = json.loads((ROOT/'data/source_inventory/download_inventory.json').read_text(encoding='utf-8'))
    sources = {r['inventory_id']: r for r in inventory}
    folder = ROOT/'data/interim/cdf_other_years'
    folder.mkdir(parents=True, exist_ok=True)
    engine = RapidOCR(rec_model_path=str(ROOT/'models/en_PP-OCRv3_rec_infer.onnx'),
                      intra_op_num_threads=2, inter_op_num_threads=2, det_limit_side_len=1684)
    rows = []
    for layout in CDF_LAYOUTS:
        source = sources[layout['source_id']]
        pdf_path = next((ROOT/'data/raw/council_documents').rglob(f"{layout['source_id']:03d}--*"))
        pdf = pdfium.PdfDocument(pdf_path)
        image = pdf[layout['page']-1].render(scale=2).to_pil()
        if layout['rotate']:
            image = image.rotate(layout['rotate'], expand=True)
        image_path = folder/f"{layout['year']}_page_{layout['page']}.png"
        image.save(image_path)
        pdf.close()
        result, _ = engine(str(image_path))
        if not result:
            raise ValueError(f'No text found: {image_path.name}')
        image_path.with_suffix('.json').write_text(json.dumps(result, ensure_ascii=False, indent=2),encoding='utf-8')
        rows.extend(group_page(result, layout, source['url']))
        print(f"Read {layout['year']} page {layout['page']}: {len(layout['rows'])} table rows", flush=True)
    df = pd.DataFrame(rows)
    df.to_csv(folder/'raw_projects.csv',sep='|',index=False)
    return df

In [13]:
raw_other_years = extract_remaining_years()
display(raw_other_years.groupby('year').size().rename('records'))

Read 2022 page 1: 16 table rows


Read 2023 page 1: 16 table rows


Read 2023 page 2: 3 table rows


Read 2024 page 1: 15 table rows


Read 2026 page 1: 6 table rows


Read 2026 page 2: 9 table rows


Read 2026 page 3: 4 table rows


year
2022    16
2023    19
2024    15
2026    19
Name: records, dtype: int64

### 3.6 Correct OCR errors and keep missing values

The following edits were read from the page images. A year and printed project number identify each correction. Text that is cut off or hidden by a stamp is noted rather than invented. Spelling differences in the source remain unless the difference is an OCR error.

In [14]:
CORRECTIONS = {}

In [15]:
# Corrections for 2022
CORRECTIONS[2022] = {5: {'project_name': 'DRILLING AND EQUIPPING OF SOLAR POWERED BOREHOLE AT HACHIBBUBA'},
 6: {'project_name': 'DRILLING AND EQUIPPING OF SOLAR POWERED BOREHOLE AT T-JUNCTION'},
 9: {'project_name': 'CONSTRUCTION OF 1X3 CLASSROOM BLOCK AT HAMBUTO PRIMARY SCHOOL',
     'project_description': 'Construction of 1x3 CRB at Hambuto Primary School',
     'ward': 'IBBWEMUNYAMA'},
 10: {'project_name': 'CONSTRUCTION OF STAFF HOUSE, ABLUTION BLOCK AND DRILLING AND EQUIPPING OF '
                      'SOLAR POWERED BOREHOLE AT VELU HEALTH POST'},
 11: {'zone': 'MANDENGA'},
 12: {'project_description': 'Construction of Semi-detached Staff House at Machavika Primary '
                             'School'},
 13: {'project_name': 'CONSTRUCTION OF 1X3 CLASSROOM BLOCK, ABLUTION BLOCK, STAFF HOUSE, DRILLING '
                      'AND EQUIPPING OF SOLAR POWERED BOREHOLE AT KATWEZELE PRIMARY SCHOOL',
      'project_description': 'Construction of 1no 1x3 CRB, 1no Staff House, 1no Ablution block and '
                             'drilling/equipping of 1no Solar powered borehole at Katwezele '
                             'Community School',
      'location': None},
 14: {'location': None},
 15: {'project_description': 'SUPPLY AND DELIVERY OF DESKS WORTH ZMW1,746,363.23',
      'location': None},
 16: {'project_name': 'CONSTRUCTION OF 1X2 CRB AT KATWEZELE PRIMARY SCHOOL',
      'location': 'KATWEZELE COMMUNITY SCHOOL'}}

In [16]:
# Corrections for 2023
CORRECTIONS[2023] = {7: {'project_description': 'CONSTRUCTION OF ABLUTION BLOCK WITH SOLAR POWERED WATER SUPPLY AT '
                            'T-JUNCTION',
     'sector': 'SANITATION',
     'ward': 'CHIRUNDU CENTRAL',
     'zone': 'MANDENGA'},
 8: {'project_name': 'CONSTRUCTION 2 OF ABLUTION BLOCKS AT MISSION MARKET AND YELLOW [text cut '
                     'off]'},
 10: {'project_description': 'PROCUREMENT OF 150 DESKS FOR ZALAUNGA PRIMARY SCHOOL'},
 11: {'project_description': 'PROCUREMENT OF 200 DESKS FOR NYANZALA PRIMARY SCHOOL'},
 13: {'project_description': 'PROCUREMENT OF 200 DESKS AT MAUNGA PRIMARY SCHOOL',
      'ward': 'IBBWEMUNYAMA',
      'zone': 'MAUNGA',
      'location': 'MAUNGA COMMUNITY SCHOOL'},
 14: {'project_name': 'PROCUREMENT OF DESKS AT 4 MILES COMMUNITY SCHOOL',
      'project_description': 'PROCUREMENT OF 100 DESKS FOR FOUR MILES COMMUNITY SCHOOL',
      'ward': 'CHIRUNDU WEST',
      'zone': 'LUSUMPUKO',
      'location': 'FOUR MILES SCHOOL'},
 15: {'ward': 'CHIRUNDU WEST', 'zone': 'CHIBENDE', 'location': 'CHIBENDE CLINIC'},
 16: {'zone': 'PAMBAZANA'},
 17: {'project_description': 'Rehabilitation of Lusitu Water System to include Chilindi, '
                             'Machavika, Chibulameenda, extension from Siamaundu to Siabulembo',
      'ward': 'LUSITU, NJAME, NGOMBE ILEDE'},
 18: {'project_description': 'DRILLING AND EQUIPPING OF SOLAR POWRED BOREHOLE AT NABBANDA CLINIC',
      'sector': 'WATER'},
 19: {'project_name': 'INSTALLATION OF WATER SUPPLY NETWORK IN CHIRUNDU CENTRAL WARD',
      'project_description': 'EXTENSION OF WATER SUPPLY IN KANENGUMBO, KADUNGA AND MANDENGA '
                             'VILLAGE',
      'sector': 'WATER'}}

In [17]:
# Corrections for 2024
CORRECTIONS[2024] = {1: {'project_name': 'CONSTRUCTION OF 1X2 CRB, INSTALLATION AND EQUIPPING OF SOLAR POWERED '
                     'BOREHOLE, CONSTRUCTION OF TOILETS, AT ZALAUNGA PRIMARY SCHOOL'},
 2: {'project_name': 'CONSTRUCTION OF 1X3 CRB AT SIKOONGO SKILLS CENTER'},
 3: {'ward': 'SIKOONGO'},
 5: {'project_name': 'CONSTRUCTION OF 1X3 CRB & ABLUTION BLOCK AT NAMABUYU SCHOOL',
     'ward': 'NJAME'},
 11: {'project_name': 'INSTALLATION OF SOLAR POWERED WATER SYSTEM IN KAPULULIRA'},
 12: {'project_name': 'CONSTRUCTION OF 1X3 CRB, ABLUTION BLOCK, INSTALLATION AND EQUIPPING OF '
                      'SOLAR POWERED BOREHOLES AT CHIPEPO COMMUNITY SCHOOL'},
 15: {'project_name': 'PROCUREMENT OF MOTORBIKES'}}

In [18]:
# Corrections for 2026
CORRECTIONS[2026] = {1: {'project_name': 'Construction of 1x3 CRB, 1 Staff house, A solar powered borehole and '
                     'Maternity Annex'},
 2: {'zone': 'Chilindi; Namabuyu',
     'location': 'Chilindi Primary School; Namabuyu Primary School',
     'work_item': 'Staff houses; 1 solar powered borehole'},
 3: {'project_name': 'Construction of sports facility (phase 2)'},
 5: {'work_item': 'Siabulembo road'},
 6: {'work_item': '1x3 CRB'},
 16: {'project_name': 'Variation (Construction of 2 courses and columns at stadium)'},
 19: {'project_name': 'Procurement of Desks'}}

In [19]:
NOTES = {
    (2022,13): 'Location text is obscured by the council stamp; left blank.',
    (2022,14): 'Stamp text was removed from the location cell; no location recovered.',
    (2022,15): 'Amount is the reported value of desk supply, not confirmed expenditure. Stamp text was removed from the location cell.',
    (2022,16): 'Project name says Primary School; location says Community School. Source wording retained.',
    (2023,8): 'Project-name text is cut off at the column edge; the description identifies Mission and Yellow Jacket markets.',
    (2023,17): 'One source entry covers multiple wards; it has not been split into three projects.',
    (2023,18): 'The location cell ends CLINI in the source. Source spelling retained.',
    (2023,19): 'The project-name ending was read with the ward column. Description says KANENGUMBO; location says KANEGUMBO. Source spellings retained.',
    (2026,2): 'One numbered entry covers two schools; sites and corresponding work items are separated by semicolons in source order.',
    (2026,8): 'Project/zone spell Shangwemu; location spells Shyangwemu. Source wording retained.',
    (2026,10): 'Project name says Mateaunga; location says Mateaungu. Source wording retained.',
    (2026,12): 'Project name spells Hachibubba; zone/location spell Hachibbuba. Source wording retained.',
    (2026,14): '2026 list entry refers to completion work on a 2024 project; it is not evidence that the work is complete.',
    (2026,15): '2026 list entry refers to completion work on a 2024 project; it is not evidence that the work is complete.',
}

In [20]:
def clean_remaining_years(raw):
    df = raw.copy()
    for year, projects in CORRECTIONS.items():
        for number, changes in projects.items():
            for column, value in changes.items():
                df.loc[(df.year == year) & (df.project_no == number), column] = value
    text_columns = ['project_name','project_description','sector','ward','zone','location','work_item']
    for column in text_columns:
        df[column] = df[column].astype('string').str.replace(r'\s+', ' ', regex=True).str.strip()
    # Repair inconsistent spacing in the same literal all-wards label.
    for column in ['ward','zone','location']:
        df[column] = df[column].str.replace(r'(?i)ALL\s*12\s*WARDS','All 12 Wards',regex=True)
    df['ward'] = df['ward'].str.replace(r'(?i)CHIRUNDU\s*-\s*WEST','Chirundu West',regex=True).str.title()
    df['sector'] = df['sector'].str.title()
    # The wording 'worth ZMW...' gives a supply value, not an allocation or payment.
    values = df['project_description'].str.extract(r'(?i)worth\s+ZMW\s*([\d,]+\.\d{2})',expand=False)
    df['amount_zmw'] = pd.to_numeric(values.str.replace(',','',regex=False), errors='coerce')
    df['amount_type'] = pd.NA
    df.loc[df.amount_zmw.notna(),'amount_type'] = 'Reported supply value'
    df['implementation_status'] = pd.NA
    df['approval_comment'] = pd.NA
    df['approval_status'] = 'Approved list'
    df['notes'] = [NOTES.get((r.year,r.project_no),'') for r in df.itertuples()]
    return df.drop_duplicates(subset=['year','project_no','source_url','source_page'])

In [21]:
other_years = clean_remaining_years(raw_other_years)
display(other_years.groupby('year').size().rename('records'))

year
2022    16
2023    19
2024    15
2026    19
Name: records, dtype: int64

### 3.7 Combine the annual lists and export

We combine the five annual lists with `pd.concat()`, sort by year and project number, and remove duplicate source rows. A project name containing “completion” is not evidence of completed work.

The one reported monetary amount is a desk-supply value in the 2022 list. It must not be interpreted as confirmed expenditure. The earlier pilot sample is not added as another dataset.

In [22]:
projects_2025 = df.copy()
projects_2025['approval_status'] = 'Approved list'
projects_2025.loc[projects_2025['project_no'] == 11, 'approval_status'] = 'Approved with scope restriction'
projects_2025['notes'] = ''
projects_2025.loc[projects_2025['project_no'] == 11, 'notes'] = 'Only procurement of the motor grader is approved in the source comment.'

combined = pd.concat([other_years, projects_2025], ignore_index=True)
columns = ['project_no', 'year', 'project_name', 'project_description', 'sector',
           'ward', 'zone', 'location', 'work_item', 'approval_status', 'approval_comment',
           'amount_zmw', 'amount_type', 'implementation_status', 'source_url', 'source_page', 'notes']
combined = combined[columns].sort_values(['year', 'project_no']).reset_index(drop=True)
combined = combined.drop_duplicates(subset=['year', 'project_no', 'source_url', 'source_page'])

print('Records by year:')
print(combined.groupby('year').size().to_string())
print('Total records:', len(combined))
print('Missing project names:', combined['project_name'].isna().sum())
print('Missing ward values:', combined['ward'].isna().sum())

output_folder = ROOT / 'data/processed/cdf_projects'
output_folder.mkdir(parents=True, exist_ok=True)
output_file = output_folder / 'db-unza26-csc4792-chirundu_cdf_approved_projects_2022_2026.csv'
try:
    combined.to_csv(output_file, sep='|', index=False, encoding='utf-8')
except PermissionError:
    # A desktop preview can hold this already-generated CSV open on Windows.
    from io import StringIO
    saved = pd.read_csv(output_file, sep='|')
    expected = pd.read_csv(StringIO(combined.to_csv(sep='|', index=False)), sep='|')
    pd.testing.assert_frame_equal(saved, expected, check_dtype=False)
    print('Existing approved-project CSV is current; Windows has it open.')
print('Saved:', output_file.name)

Records by year:
year
2022    16
2023    19
2024    15
2025    14
2026    19
Total records: 83
Missing project names: 0
Missing ward values: 3
Existing approved-project CSV is current; Windows has it open.
Saved: db-unza26-csc4792-chirundu_cdf_approved_projects_2022_2026.csv


## 4. Extract Annual Budgets

The budget PDFs have selectable text. We read them with pdfplumber and keep the current-year column from each report. Programme allocations, CDF subprogrammes, revenue estimates and printed council totals go into separate tables.

The selected 2024 budget has no detailed revenue schedule. The 2025 Capital Grant Credit label is visible in the image but absent from extracted text; the correction below restores it and records why.

In [23]:
# Source id, year, programme page, CDF page, revenue pages
BUDGETS = [(104, 2022, 4, 7, [2, 3, 4]), (34, 2023, 5, 8, [3, 4, 5]),
           (33, 2024, 3, 4, []), (32, 2025, 7, 8, [3, 4]),
           (30, 2026, 7, 8, [3, 4, 5])]

BUDGET_NUMBER_PATTERN = r'(?:\([\d,]+\)|[\d,]+|-)'

def parse_budget_amount(value):
    if value == '-':
        return None
    return float(value.replace(',', '').replace('(', '-').replace(')', ''))

In [24]:
def extract_budgets():
    sources = {r['inventory_id']: r for r in json.loads((ROOT/'data/source_inventory/download_inventory.json').read_text(encoding='utf-8'))}
    programmes, cdf, revenues, totals = [], [], [], []
    for sid, year, pp, cp, revenue_pages in BUDGETS:
        source = sources[sid]
        pdf_path = next((ROOT/'data/raw/council_documents').rglob(f'{sid:03d}--*'))
        with pdfplumber.open(pdf_path) as pdf:
            def lines(page):
                return pdf.pages[page-1].extract_text().replace('(cid:9)', '').splitlines()
            active = False
            previous = ''
            for line in lines(pp):
                if 'Allocation by Programme' in line and not line.startswith('Figure'):
                    active = True
                    continue
                if not active:
                    continue
                if 'Head Total' in line:
                    total_line = previous if line == 'Head Total' else line
                    value = re.findall(BUDGET_NUMBER_PATTERN, total_line)[-1]
                    totals.append(dict(year=year, amount_zmw=parse_budget_amount(value), source_url=source['url'], source_page=pp))
                    break
                match = re.fullmatch(r'(\d{1,2})\s*([A-Za-z].*?)\s+('+BUDGET_NUMBER_PATTERN+r')\s+('+BUDGET_NUMBER_PATTERN+r')\s+('+BUDGET_NUMBER_PATTERN+r')', line)
                if match:
                    code, label, _, _, value = match.groups()
                    programmes.append(dict(year=year, programme_code=code, programme=label, amount_zmw=parse_budget_amount(value), source_url=source['url'], source_page=pp))
                previous = line
            for line in lines(cp):
                match = re.fullmatch(r'(77[9]|78[0-3])\s+(.+?)\s+('+BUDGET_NUMBER_PATTERN+r')\s+('+BUDGET_NUMBER_PATTERN+r')\s+('+BUDGET_NUMBER_PATTERN+r')\s+('+BUDGET_NUMBER_PATTERN+r')\s+('+BUDGET_NUMBER_PATTERN+r')', line)
                if match:
                    code, label, *values = match.groups()
                    cdf.append(dict(year=year, subprogramme_code=code, subprogramme=label, amount_zmw=parse_budget_amount(values[-1]), source_url=source['url'], source_page=cp))
            category = None
            for page in revenue_pages:
                for line in lines(page):
                    if 'BUDGET SUMMARY' in line:
                        break
                    cat = re.fullmatch(r'(\d{2})\s+([A-Za-z].*)', line)
                    if cat and not re.search(r'\d', cat[2]):
                        category = cat[2]
                    elif re.fullmatch(r'\d{2}', line):
                        category = None
                    match = re.fullmatch(r'(\d{3})\s+(.*?)\s*('+BUDGET_NUMBER_PATTERN+r')\s+('+BUDGET_NUMBER_PATTERN+r')\s+('+BUDGET_NUMBER_PATTERN+r')', line)
                    if match:
                        code, label, value, _, _ = match.groups()
                        note = ''
                        if not label:
                            note = 'Revenue description is blank in the source table.'
                        if sid == 32 and page == 4 and code == '002' and not label:
                            label = 'Capital Grant Credit ($300k)'
                            note = 'Description transcribed from the page image because PDF text extraction omitted it. Category name is blank in the source.'
                        revenues.append(dict(year=year, category=category, revenue_code=code, revenue_description=label or None, amount_zmw=parse_budget_amount(value), source_url=source['url'], source_page=page, notes=note))
    frames = {}
    for name, rows in [('budget_programmes', programmes), ('cdf_budget_subprogrammes', cdf), ('budget_revenue', revenues), ('budget_totals', totals)]:
        df = pd.DataFrame(rows).drop_duplicates()
        for col in df:
            if col not in ['year', 'amount_zmw', 'source_page']:
                df[col] = df[col].astype('string').str.strip().replace('', pd.NA)
        frames[name] = df
    return frames

In [25]:
budget_tables = extract_budgets()
display(pd.DataFrame({name: table.groupby('year').size() for name, table in budget_tables.items()}))
display(budget_tables['budget_programmes'].head())
display(budget_tables['cdf_budget_subprogrammes'])

,budget_programmes,cdf_budget_subprogrammes,budget_revenue,budget_totals
year,,,,
2022,11,5,63.0,1
2023,12,5,55.0,1
2024,15,4,NaN,1
2025,16,4,57.0,1
2026,16,4,56.0,1


,year,programme_code,programme,amount_zmw,source_url,source_page
0,2022,1,Constituency Development,25700000.0,http://www.chirunducouncil.gov.zm/wp-content/uploads/2024/11/Chiru...,4
1,2022,2,Local Governance,825680.0,http://www.chirunducouncil.gov.zm/wp-content/uploads/2024/11/Chiru...,4
2,2022,3,Integrated Development Planning,3855287.0,http://www.chirunducouncil.gov.zm/wp-content/uploads/2024/11/Chiru...,4
3,2022,4,Economic and Business Development,1188000.0,http://www.chirunducouncil.gov.zm/wp-content/uploads/2024/11/Chiru...,4
4,2022,5,Public Health and Environmental Protection,576235.0,http://www.chirunducouncil.gov.zm/wp-content/uploads/2024/11/Chiru...,4


,year,subprogramme_code,subprogramme,amount_zmw,source_url,source_page
0,2022,779,Community Capital Projects,15420000.0,http://www.chirunducouncil.gov.zm/wp-content/uploads/2024/11/Chiru...,7
1,2022,780,Youth Empowerment,2570000.0,http://www.chirunducouncil.gov.zm/wp-content/uploads/2024/11/Chiru...,7
2,2022,781,Women Empowerment,2570000.0,http://www.chirunducouncil.gov.zm/wp-content/uploads/2024/11/Chiru...,7
3,2022,782,Secondary School Barsaries,2570000.0,http://www.chirunducouncil.gov.zm/wp-content/uploads/2024/11/Chiru...,7
4,2022,783,Skills Development Barsaries,2570000.0,http://www.chirunducouncil.gov.zm/wp-content/uploads/2024/11/Chiru...,7
5,2023,779,Community Capital Projects,16980000.0,http://www.chirunducouncil.gov.zm/wp-content/uploads/2024/11/Chiru...,8
6,2023,780,Youth Empowerment,2830000.0,http://www.chirunducouncil.gov.zm/wp-content/uploads/2024/11/Chiru...,8
7,2023,781,Women Empowerment,2830000.0,http://www.chirunducouncil.gov.zm/wp-content/uploads/2024/11/Chiru...,8
8,2023,782,Secondary School Barsaries,2830000.0,http://www.chirunducouncil.gov.zm/wp-content/uploads/2024/11/Chiru...,8
9,2023,783,Skills Development Barsaries,2830000.0,http://www.chirunducouncil.gov.zm/wp-content/uploads/2024/11/Chiru...,8


## 5. Extract Annual Financial Statements

We use the main cash, final-budget comparison, CDF and LGEF tables for 2022–2024. The 2024 report also contains ZDSP capital-grant and sector-grant statements. These signed PDFs require OCR.

### 5.1 Read the selected pages

The page list is explicit. Existing OCR results can be reused; otherwise the following function renders and reads the PDF pages.

In [26]:
FINANCIAL_PAGES = {39: [11, 12, 13, 14], 38: [11, 12, 13, 14], 35: [12, 13, 14, 15, 16, 17]}

def extract_financial_pages():
    sources = {r['inventory_id']: r for r in json.loads((ROOT/'data/source_inventory/download_inventory.json').read_text(encoding='utf-8'))}
    folder = ROOT/'data/interim/finance'
    folder.mkdir(parents=True, exist_ok=True)
    engine = None
    for sid, pages in FINANCIAL_PAGES.items():
        pdf_path = next((ROOT/'data/raw/council_documents').rglob(f'{sid:03d}--*'))
        pdf = pdfium.PdfDocument(pdf_path)
        for page in pages:
            target = folder/f'{sid}_page_{page}.json'
            if target.exists():
                continue
            if engine is None:
                engine = RapidOCR(rec_model_path=str(ROOT/'models/en_PP-OCRv3_rec_infer.onnx'), intra_op_num_threads=2, inter_op_num_threads=2, det_limit_side_len=1684)
            path = target.with_suffix('.png')
            pdf[page-1].render(scale=2).to_pil().save(path)
            result, _ = engine(str(path))
            target.write_text(json.dumps(result, ensure_ascii=False, indent=2), encoding='utf-8')
            print(f'Read financial statement {sid}, PDF page {page}', flush=True)
        pdf.close()

### 5.2 Group and clean the financial rows

The layouts specify statement names, row limits, column positions and the small slope of each scan. Only the current reporting year is extracted. Dashes remain missing, printed zeroes remain zero, and parentheses indicate negative values. OCR corrections are kept inside the function alongside the affected rows.

In [27]:
# Source, page, year, statement, row limits, column limits and slope
FINANCIAL_LAYOUTS = [
    (39,11,2022,'Council',290,1035,590,680,865,0.005),
    (39,13,2022,'LGEF',302,925,560,680,870,0.005),
    (39,14,2022,'CDF',300,740,520,620,810,0.005),
    (38,11,2023,'Council',240,960,590,700,900,0.008),
    (38,13,2023,'LGEF',258,905,560,700,900,0.010),
    (38,14,2023,'CDF',276,795,620,700,880,0.019),
    (35,12,2024,'Council',256,960,600,700,875,-0.006),
    (35,14,2024,'LGEF',289,996,510,640,850,-0.003),
    (35,15,2024,'CDF',287,945,615,700,875,0.003),
    (35,16,2024,'ZDSP capital grant',252,625,525,640,855,-0.008),
    (35,17,2024,'Sector grant',280,653,530,640,855,-0.005),
    (39,12,2022,'Budget comparison',342,985,440,720,825,0.003),
    (38,12,2023,'Budget comparison',304,906,455,725,815,0.010),
    (35,13,2024,'Budget comparison',340,1040,410,685,790,-0.004),
]

In [28]:
def parse_amount(text):
    if not text or text.strip() in ['-', '—']:
        return None
    text = text.replace(' ', '')
    # OCR sometimes uses a full stop for a thousands separator.
    text = re.sub(r'\.(?=\d{3}(?:\D|$))', ',', text)
    if not re.fullmatch(r'-?\(?[\d,]+(?:\.\d{1,2})?\)?', text):
        raise ValueError(f'Please review financial amount: {text}')
    return float(text.replace(',', '').replace('(', '-').replace(')', ''))

In [29]:
def extract_financial_tables():
    sources = {r['inventory_id']:r for r in json.loads((ROOT/'data/source_inventory/download_inventory.json').read_text(encoding='utf-8'))}
    financial, comparisons = [], []
    for sid, page, year, statement, top, bottom, label_right, left, right, slope in FINANCIAL_LAYOUTS:
        data = json.loads((ROOT/f'data/interim/finance/{sid}_page_{page}.json').read_text(encoding='utf-8'))
        tokens = []
        for box, text, confidence in data:
            x = sum(p[0] for p in box)/4
            y = sum(p[1] for p in box)/4 - slope*(x-250)
            if top-8 <= y <= bottom:
                tokens.append((x,y,text))
        labels = []
        for x,y,text in sorted(tokens, key=lambda t:t[1]):
            if x >= label_right:
                continue
            if labels and abs(labels[-1][0]-y)<7:
                labels[-1][1] += ' '+text
            else:
                labels.append([y,text])
        if (sid,page)==(39,11):
            labels += [[948,'Increase/(decrease) in Cash'],[975,'Foreign Exchange Losses'],[1001,'Cash at beginning of the year'],[1027,'Cash at the end of the year']]
        if (sid,page)==(38,14):
            labels = [[y,t+' Bursaries' if 590<y<605 else t] for y,t in labels if t!='Bursaries']
        section, subsection = '', ''
        for y,label in sorted(labels):
            label = re.sub(r'\s+\d+\([a-z]\)$', '', label).strip()
            label = label.replace('TOTALRECEIPTS','TOTAL RECEIPTS').replace('TOTALPAYMENTS','TOTAL PAYMENTS')
            label = label.replace('Commerc ial','Commercial').replace('Financ ial','Financial').replace('Fees ard','Fees and').replace('Emolurrents','Emoluments').replace('Asscts','Assets').replace('Non-financial l Assets','Non-financial Assets')
            label = label.replace('Non-financialAssets','Non-financial Assets').replace('ofthe','of the').replace('the-end','the end')
            label = label.replace('Increase/(decreasein', 'Increase/(decrease) in').replace('Increase/(Decre ase in','Increase/(Decrease) in').replace('Increase/(Decrease in','Increase/(Decrease) in')
            if label in ['RECEIPTS','PAYMENTS']:
                section = label.lower()
                subsection = ''
                continue
            if 'Expenditure Payments' in label:
                subsection = 'operational' if 'Operational' in label else 'capital'
                continue
            row_type = 'total' if label.startswith('TOTAL') else 'subtotal' if label=='Sub-Total' else 'detail'
            if 'Cash' in label or 'cash' in label or 'Foreign Exchange' in label:
                section, subsection, row_type = 'cash balance', '', 'balance'
            def read_column(a,b):
                found = [(x,t) for x,yy,t in tokens if a<=x<b and abs(yy-y)<11]
                return ' '.join(t for x,t in sorted(found))
            raw = read_column(left,right)
            note = ''
            if (sid,page)==(38,14) and 'Increase/' in label:
                raw = '-3,340,270'
                note = 'Restored the minus sign missed by OCR after checking the page image.'
            if (sid,page)==(38,14) and '31 December 2022' in label:
                note = 'Source closing-balance label says 2022 within the 2023 statement; report year retained.'
            if (sid,page)==(35,15) and 'Secondary Boarding' in label:
                label = 'Secondary Boarding Schools and Skills Dev Bursaries'
            row = dict(year=year, statement=statement, section=section, subsection=subsection or None, row_type=row_type, description=label,
                       amount_zmw=parse_amount(raw), source_url=sources[sid]['url'], source_page=page, notes=note or None)
            if statement=='Budget comparison':
                # Use final budget and actuals: original estimates can contain cents,
                # while the comparison's final budgets are printed as whole Kwacha.
                a,b = (600,720) if sid in [39,38] else (590,680)
                row['final_budget_zmw'] = parse_amount(read_column(a,b))
                row['actual_amount_zmw'] = row.pop('amount_zmw')
                comparisons.append(row)
            else:
                financial.append(row)
    return {'financial_statements':pd.DataFrame(financial).drop_duplicates(), 'budget_vs_actual':pd.DataFrame(comparisons).drop_duplicates()}

In [30]:
extract_financial_pages()
financial_tables = extract_financial_tables()
financial_df = financial_tables['financial_statements']
comparison_df = financial_tables['budget_vs_actual']
display(financial_df.groupby(['year', 'statement']).size().rename('rows'))
display(financial_df.loc[(financial_df['statement'] == 'CDF') & (financial_df['row_type'] == 'total'),
                         ['year', 'section', 'description', 'amount_zmw']])

year  statement         
2022  CDF                   13
      Council               25
      LGEF                  20
2023  CDF                   16
      Council               25
      LGEF                  20
2024  CDF                   18
      Council               26
      LGEF                  20
      Sector grant          12
      ZDSP capital grant    12
Name: rows, dtype: int64

,year,section,description,amount_zmw
47,2022,receipts,TOTAL RECEIPTS,23739911.0
54,2022,payments,TOTAL PAYMENTS,6518164.0
106,2023,receipts,TOTAL Funding,27503433.0
115,2023,payments,TOTAL PAYMENTS,30843703.0
168,2024,receipts,TOTAL Funding,10675337.0
179,2024,payments,TOTAL PAYMENTS,24885731.0


### 5.3 Compare the final budget with actual amounts

We calculate `actual - final budget`. A positive payment difference means spending exceeded the final budget. Missing inputs give a missing result.

Detail rows, subtotals, totals and cash balances are labelled separately. Do not sum those row types together, or add council totals to the CDF/LGEF figures already included in them. Published differences between totals and line items are retained.

For example, the 2023 cash statement and comparison differ by K1 on Commercial Venture and Other Receipts. The 2024 CDF statement distinguishes K10 million funding from loan repayments of K675,337. Its total receipts are not a fresh government allocation. The 2023 CDF closing-balance label says 2022 in the source; a note preserves that issue.

In [31]:
comparison_df['calculated_variance_zmw'] = comparison_df['actual_amount_zmw'] - comparison_df['final_budget_zmw']
display(comparison_df.loc[comparison_df['row_type'] == 'total',
                          ['year', 'description', 'final_budget_zmw', 'actual_amount_zmw', 'calculated_variance_zmw']])
display(financial_df.isna().sum().rename('missing_values'))

budget_review = budget_tables['budget_totals'][['year', 'amount_zmw']].set_index('year').rename(columns={'amount_zmw': 'printed_total_zmw'})
budget_review['sum_of_programmes_zmw'] = budget_tables['budget_programmes'].groupby('year')['amount_zmw'].sum()
display(budget_review)

,year,description,final_budget_zmw,actual_amount_zmw,calculated_variance_zmw
11,2022,TOTAL RECEIPTS,50030983.0,43982428.0,-6048555.0
20,2022,TOTAL PAYMENTS,50030983.0,28353881.0,-21677102.0
33,2023,TOTAL RECEIPTS,53065765.0,57868462.0,4802697.0
42,2023,TOTAL PAYMENTS,53065765.0,61291747.0,8225982.0
56,2024,TOTAL RECEIPTS,79762187.0,54935597.0,-24826590.0
65,2024,TOTAL PAYMENTS,79762187.0,64029018.0,-15733169.0


year             0
statement        0
section          0
subsection     165
row_type         0
description      0
amount_zmw      68
source_url       0
source_page      0
notes          205
Name: missing_values, dtype: int64

,printed_total_zmw,sum_of_programmes_zmw
year,,
2022,50030983.0,50030984.0
2023,53065765.0,53065765.0
2024,79762187.0,79762188.0
2025,103929416.0,103929415.0
2026,112489567.0,112489566.0


### 5.4 Export the budget and annual financial tables

Each table is saved with the required filename prefix and a pipe separator. Supporting notes and individual transactions are outside this extraction so far.

In [32]:
finance_folder = ROOT / 'data/processed/finance'
finance_folder.mkdir(parents=True, exist_ok=True)
finance_tables = {**budget_tables, **financial_tables}
saved_files = []
for name, table in finance_tables.items():
    table = table.drop_duplicates().reset_index(drop=True)
    filename = f'db-unza26-csc4792-chirundu_{name}.csv'
    table.to_csv(finance_folder / filename, sep='|', encoding='utf-8', index=False)
    saved_files.append({'file': filename, 'rows': len(table)})
display(pd.DataFrame(saved_files))

,file,rows
0,db-unza26-csc4792-chirundu_budget_programmes.csv,70
1,db-unza26-csc4792-chirundu_cdf_budget_subprogrammes.csv,22
2,db-unza26-csc4792-chirundu_budget_revenue.csv,231
3,db-unza26-csc4792-chirundu_budget_totals.csv,5
4,db-unza26-csc4792-chirundu_financial_statements.csv,207
5,db-unza26-csc4792-chirundu_budget_vs_actual.csv,66


## 6. Extract CDF Performance and Project Progress

We use inventory 31 for the selectable January–June 2025 financial table, inventory 36 for the signed report's CDF indicators, and inventory 75 for named Q1 2023 procurement progress. Inventory 37 appears to duplicate the signed report and is not added again. Inventory 88 is an empty template, and inventory 90 is another approval list.

### 6.1 Read scanned performance pages

The procurement pages are sideways, so we rotate them before OCR. Previously saved OCR results are reused when present.

In [33]:
PERFORMANCE_PAGES = {36: [4], 75: [2, 3, 8, 9]}

def extract_performance_pages():
    folder = ROOT/'data/interim/performance'
    folder.mkdir(parents=True, exist_ok=True)
    engine = None
    for sid, pages in PERFORMANCE_PAGES.items():
        pdf = pdfium.PdfDocument(next((ROOT/'data/raw/council_documents').rglob(f'{sid:03d}--*')))
        for page in pages:
            target = folder/f'{sid}_page_{page}.json'
            if target.exists():
                continue
            image = pdf[page-1].render(scale=2).to_pil()
            if sid == 75:
                image = image.rotate(90, expand=True)
            image.save(target.with_suffix('.png'))
            if engine is None:
                engine = RapidOCR(rec_model_path=str(ROOT/'models/en_PP-OCRv3_rec_infer.onnx'), intra_op_num_threads=2, inter_op_num_threads=2, det_limit_side_len=1684)
            result, _ = engine(str(target.with_suffix('.png')))
            target.write_text(json.dumps(result, ensure_ascii=False, indent=2), encoding='utf-8')
            print(f'Read source {sid}, page {page}', flush=True)
        pdf.close()

### 6.2 Helpers for source records, amounts and OCR cells

These small functions load the source inventory, clean an amount and collect words inside one table cell. Dates and money are cleaned without changing their meaning.

In [34]:
def load_sources():
    return {r['inventory_id']: r for r in json.loads((ROOT/'data/source_inventory/download_inventory.json').read_text(encoding='utf-8'))}

def parse_performance_number(text):
    text = re.sub(r'\s+', '', str(text or ''))
    if text in ['', '-', 'None']:
        return None
    return float(text.replace(',', '').replace('(', '-').replace(')', ''))

def read_cell(sid, page, left, right, top, bottom):
    tokens = json.loads((ROOT/f'data/interim/performance/{sid}_page_{page}.json').read_text(encoding='utf-8'))
    words = []
    for box, text, _ in tokens:
        x, y = (sum(p[i] for p in box)/4 for i in [0, 1])
        if left <= x < right and top <= y < bottom:
            words.append((round(y/10), x, text))
    return ' '.join(t for _, _, t in sorted(words)).strip()

In [35]:
extract_performance_pages()
print('Performance OCR pages are ready.')

Performance OCR pages are ready.


### 6.3 Extract half-year financial performance

These are six-month actuals against an annual budget. The source uses `budget - actual` for its variance, opposite to our annual comparison calculation. Its percentage column contains spreadsheet errors, which are kept as text with notes. The K0.01 net budget and its implausible percentage must not be interpreted as meaningful performance.

In [36]:
def extract_half_year_finances():
    source = load_sources()[31]
    with pdfplumber.open(next((ROOT/'data/raw/council_documents').rglob('031--*'))) as pdf:
        tables = pdf.pages[0].extract_tables()
    rows = []
    for section, table in zip(['receipts', 'payments'], tables[:2]):
        category = None
        for code, label, budget, actual, variance, performance in table:
            if section == 'receipts' and code in ['1', '2', '3']:
                category = label
                continue
            if budget in [None, '', 'Budget', 'Budget\na\nZMW', 'a', 'ZMW']:
                continue
            label = label or code
            if label in ['Receipts', 'Payments']:
                continue
            row_type = 'subtotal' if label=='Sub - total' else 'total' if label.startswith('Total') else 'detail'
            match = re.match(r'(\d+\.\d+)\s+(.+)', label)
            if match:
                code, label = match.groups()
            note = None
            if label == 'Others OSR':
                note = 'Actual is printed as 355,200 with no cents; printed variance implies a 0.10 difference. Source figures retained.'
            rows.append(dict(period_start='2025-01-01', period_end='2025-06-30', section=section,
                             category=category if row_type!='total' else None, row_type=row_type,
                             source_code=code if row_type=='detail' else None, description=label,
                             annual_budget_zmw=parse_performance_number(budget), actual_to_date_zmw=parse_performance_number(actual),
                             reported_variance_zmw=parse_performance_number(variance), performance_as_printed=performance,
                             source_url=source['url'], source_page=1, notes=note))
    # Preserve the source's net row, including its anomalous percentage, rather than fixing it silently.
    label, budget, actual, variance, performance = tables[2][0]
    rows.append(dict(period_start='2025-01-01', period_end='2025-06-30', section='net balance',
                     category=None, row_type='balance', source_code=None, description=label,
                     annual_budget_zmw=parse_performance_number(budget), actual_to_date_zmw=parse_performance_number(actual),
                     reported_variance_zmw=parse_performance_number(variance), performance_as_printed=performance,
                     source_url=source['url'], source_page=1,
                     notes='The source prints a 0.01 net budget and an anomalous percentage. Do not interpret that percentage as meaningful performance.'))
    return pd.DataFrame(rows).drop_duplicates()

In [37]:
half_year_df = extract_half_year_finances()
display(half_year_df.loc[half_year_df['row_type'].isin(['total', 'balance']),
                         ['description', 'annual_budget_zmw', 'actual_to_date_zmw', 'reported_variance_zmw']])
display(half_year_df.loc[half_year_df['description'] == 'Constituency Development Fund'])

,description,annual_budget_zmw,actual_to_date_zmw,reported_variance_zmw
20,Total,1.039294e+08,41644088.08,62285327.77
29,Total payments,1.039294e+08,43832931.75,60096484.10
30,Net Budget Performance,1.000000e-02,-2188843.67,2188843.68


,period_start,period_end,section,category,row_type,source_code,description,annual_budget_zmw,actual_to_date_zmw,reported_variance_zmw,performance_as_printed,source_url,source_page,notes
4,2025-01-01,2025-06-30,receipts,National Support,detail,1.5,Constituency Development Fund,36058150.6,20287381.52,15770769.08,56%,http://www.chirunducouncil.gov.zm/wp-content/uploads/2025/12/2025-...,1,NaN


### 6.4 Extract CDF indicators

The CDF block on page 4 contains seven indicators. Heading rows are excluded. We retain targets, actual values, units and comments. The reported 20% implementation rate is an aggregate programme indicator, not a completion percentage for each approved project. A clipped word in A11 is completed as “sponsored” and noted.

In [38]:
def extract_cdf_indicators():
    # Only the CDF-related block A1-A11 on PDF page 4. Heading rows are not observations.
    bounds = [('A2',280,329,'Community projects'), ('A4',354,379,'Empowerment'),
              ('A5',379,405,'Empowerment'), ('A7',428,452,'Monitoring and evaluation'),
              ('A8',452,479,'Monitoring and evaluation'), ('A10',502,526,'Bursaries'),
              ('A11',526,552,'Bursaries')]
    rows = []
    for code, top, bottom, programme in bounds:
        label = read_cell(36,4,300,818,top,bottom)
        if code=='A11':
            # The source clips the end of this label at the column border.
            label = 'Percentage of approved Secondary bursaries applicants sponsored'
        rows.append(dict(period_start='2025-01-01', period_end='2025-06-30', indicator_code=code,
                         programme=programme, indicator=label,
                         unit='percent' if label.startswith('Percentage') else 'count',
                         target=parse_performance_number(read_cell(36,4,818,921,top,bottom)),
                         actual=parse_performance_number(read_cell(36,4,921,1045,top,bottom)),
                         reported_variance=parse_performance_number(read_cell(36,4,1045,1148,top,bottom)),
                         comment=read_cell(36,4,1148,1560,top,bottom),
                         source_url=load_sources()[36]['url'], source_page=4,
                         notes='Clipped final word completed from context.' if code=='A11' else None))
    return pd.DataFrame(rows).drop_duplicates()

In [39]:
indicators_df = extract_cdf_indicators()
display(indicators_df[['indicator', 'unit', 'target', 'actual', 'reported_variance', 'comment']])

,indicator,unit,target,actual,reported_variance,comment
0,Percentage of Community Projects implemented,percent,100.0,20.0,80.0,"Funding is delayed, while other Projects are under the procurement..."
1,Percentage of approved grant applicants empowered,percent,100.0,100.0,0.0,100% of approved Grant applicants empowered
2,Percentage of approved loan applicants empowered,percent,100.0,0.0,100.0,The process is with the bank
3,Number of monitoring and evaluation activities conducted,count,8.0,2.0,6.0,Two Evaluations Conducted
4,Number of project appraisals carried out,count,2.0,1.0,1.0,One Project Appraisal activity was Carried Out
5,Percentage of approved Skills bursaries applicants sponsored,percent,100.0,67.0,33.0,Not all students reported
6,Percentage of approved Secondary bursaries applicants sponsored,percent,100.0,44.0,56.0,Not all students reported


### 6.5 Extract named project progress

The Q1 2023 report lists five CDF procurements in progress and three signed contracts. We join descriptions continuing onto another page. Hamunjo's 61.25% figure was restored from the image after OCR merged it with the manager column.

Estimated amounts and contract values are not actual payments. The execution figures are historical Q1 2023 observations, not current statuses.

In [40]:
def extract_project_progress():
    rows = []
    # The first five records are in Part A: Procurements in progress (pages 2-3).
    for i, (page, top, bottom) in enumerate([(2,684,815),(2,816,897),(2,898,1031),(2,1032,1088),(3,231,368)], 1):
        label = read_cell(75,page,425,580,top,bottom)
        if i==4:
            label += ' '+read_cell(75,3,425,580,149,230)
        row = dict(progress_id=f'2023Q1-P{i}', description=label,
                   tender_or_contract_number=read_cell(75,page,200,425,top,bottom).replace(' ', ''),
                   procurement_method=read_cell(75,page,580,735,top,bottom),
                   source_of_funds=read_cell(75,page,735,865,top,bottom).replace(' ', ''),
                   approval_date=read_cell(75,page,1138,1290,top,bottom),
                   amount_zmw=parse_performance_number(read_cell(75,page,1290,1450,top,bottom)), amount_type='Estimated amount',
                   reported_stage='Procurements in progress', execution_percent=None,
                   contractor=None, contract_signed_date=None, source_page=page,
                   source_page_end=3 if i==4 else page, notes=None)
        rows.append(row)
    # Part C: Contracts signed. Values wrap over multiple lines in narrow cells.
    for i, (top, bottom) in enumerate([(602,683),(684,868),(869,1086)], 6):
        label = read_cell(75,8,302,466,top,bottom)
        if i==8:
            label = label.replace('Constructiono f', 'Construction of') + ' school'
        pct = read_cell(75,8,1315,1480,top,bottom)
        note = None
        if i==8:
            pct = '61.25%'
            note = 'OCR merged execution percentage with the neighbouring manager cell; 61.25% restored from image. Description continues with school on page 9.'
        rows.append(dict(progress_id=f'2023Q1-P{i}', description=label,
                         tender_or_contract_number=read_cell(75,8,170,302,top,bottom).replace(' ', ''),
                         procurement_method=None, source_of_funds=read_cell(75,8,674,795,top,bottom).replace(' ', ''),
                         approval_date=read_cell(75,8,795,933,top,bottom),
                         amount_zmw=parse_performance_number(read_cell(75,8,1045,1168,top,bottom)), amount_type='Contract value',
                         reported_stage='Contracts signed', execution_percent=parse_performance_number(pct.replace('%','')),
                         contractor=read_cell(75,8,466,674,top,bottom),
                         contract_signed_date=read_cell(75,8,933,1045,top,bottom).replace(' ', ''),
                         source_page=8, source_page_end=9 if i==8 else 8, notes=note))
    df = pd.DataFrame(rows)
    df['period_start'], df['period_end'], df['report_date'] = '2023-01-01', '2023-03-31', '2023-04-15'
    df['source_url'] = load_sources()[75]['url']
    for col in ['approval_date','contract_signed_date']:
        df[col] = pd.to_datetime(df[col],format='%d/%m/%Y',errors='raise').dt.strftime('%Y-%m-%d')
    df['description'] = df['description'].str.replace('1x3crbat','1x3crb at',regex=False).str.replace(r'\s+', ' ', regex=True).str.strip()
    return df.drop_duplicates()

### 6.6 Link progress to approved projects

The six links below were reviewed using location and compatible work descriptions. They are inferences, not official shared identifiers. Velu and Lusitu East remain unlinked because the descriptions do not establish the same scope. Tender numbers repeat and cannot uniquely identify a project. Linked ward names come from the approved list.

In [41]:
progress_raw_df = extract_project_progress()

approved = pd.read_csv(ROOT/'data/processed/cdf_projects/db-unza26-csc4792-chirundu_cdf_approved_projects_2022_2026.csv',sep='|')
# Reviewed location and work descriptions, not fuzzy name matching.
links = {'2023Q1-P1':4,'2023Q1-P4':3,'2023Q1-P5':9,'2023Q1-P6':7,'2023Q1-P7':12,'2023Q1-P8':1}
df = progress_raw_df.copy()
df['approved_project_no'] = df['progress_id'].map(links).astype('Int64')
df['approved_year'] = pd.Series(2022,index=df.index).where(df['approved_project_no'].notna()).astype('Int64')
df['match_note'] = 'Inferred from matching location and compatible construction scope; the procurement report does not give ward names.'
df.loc[df.progress_id=='2023Q1-P2','match_note'] = 'Unlinked: Velu location matches a 2022 entry but the scope is less specific; confirm before joining.'
df.loc[df.progress_id=='2023Q1-P3','match_note'] = 'Unlinked: no matching Lusitu East classroom approval in the selected annual lists. The 2024 entry is a staff house.'
lookup = approved[['year','project_no','project_name','ward','source_url','source_page']].rename(columns={'year':'approved_year','project_no':'approved_project_no','project_name':'approved_project_name','ward':'approved_ward','source_url':'approved_source_url','source_page':'approved_source_page'})
progress_df = df.merge(lookup,on=['approved_year','approved_project_no'],how='left',validate='many_to_one')

display(progress_df[['description', 'execution_percent', 'amount_type', 'approved_year', 'approved_project_no']])
print('Linked records:', progress_df['approved_project_no'].notna().sum())

,description,execution_percent,amount_type,approved_year,approved_project_no
0,Construction of 1x3crb at siabulembo community school,NaN,Estimated amount,2022,4
1,Construction of velu rural health post,NaN,Estimated amount,<NA>,<NA>
2,Construction of 1x3crb at Lusitu east primary school,NaN,Estimated amount,<NA>,<NA>
3,Construction of 1x3crb at chisamu primary school,NaN,Estimated amount,2022,3
4,Construction of 1x3crb at hambuto primary school,NaN,Estimated amount,2022,9
5,Construction of 1x3crb in munenga,65.00,Contract value,2022,7
6,Construction of a semi detatched staff house at Machavika primary ...,70.00,Contract value,2022,12
7,"Construction of 1x3crb,staff house and solar powered borehole at h...",61.25,Contract value,2022,1


Linked records: 6


## 7. Save the Performance Tables

We retain reporting periods, source URLs and page numbers. Original approvals are not overwritten with historical procurement statuses.

In [42]:
performance_folder = ROOT / 'data/processed/performance'
performance_folder.mkdir(parents=True, exist_ok=True)
performance_tables = {'half_year_finances_2025': half_year_df,
                      'cdf_performance_indicators_2025': indicators_df,
                      'cdf_project_progress_2023_q1': progress_df}
saved_performance = []
for name, table in performance_tables.items():
    table = table.drop_duplicates().reset_index(drop=True)
    filename = f'db-unza26-csc4792-chirundu_{name}.csv'
    table.to_csv(performance_folder / filename, sep='|', encoding='utf-8', index=False)
    saved_performance.append({'file': filename, 'rows': len(table)})
display(pd.DataFrame(saved_performance))

,file,rows
0,db-unza26-csc4792-chirundu_half_year_finances_2025.csv,31
1,db-unza26-csc4792-chirundu_cdf_performance_indicators_2025.csv,7
2,db-unza26-csc4792-chirundu_cdf_project_progress_2023_q1.csv,8
